#  Study Design Classification — Validation

Validates `class.study_design` (L1/L2/L4/L5) and `class.innovative_features` (L3).

**Checks:**
- Distribution of each classification level
- Innovative feature counts by type
- Spot-check examples per feature type for precision

In [1]:
import sys
sys.path.insert(0, '..')

import pandas as pd
from config.settings import get_duckdb_connection

conn = get_duckdb_connection()
print("Connected to DuckDB")

Connected to DuckDB


## 1. Study Design Classification Overview

In [2]:
total = conn.execute("SELECT COUNT(*) FROM class.study_design").fetchone()[0]
print(f"Total studies classified: {total:,}")

Total studies classified: 590,350


### L1: Study Type

In [3]:
# L1 (study_type) is no longer carried in class.study_design — it's read straight
# off enriched.studies (Phase 7C contract; the copy was dropped 2026-06-19).
conn.execute("""
    SELECT study_type, COUNT(*) AS count
    FROM enriched.studies
    GROUP BY study_type
    ORDER BY count DESC
""").fetchdf()

,study_type,count
0,INTERVENTIONAL,450483
1,OBSERVATIONAL,137835
2,EXPANDED_ACCESS,1056
3,NaN,976


### L2: Design Architecture

In [4]:
conn.execute("""
    SELECT design_architecture, COUNT(*) AS count
    FROM class.study_design
    WHERE design_architecture IS NOT NULL
    GROUP BY design_architecture
    ORDER BY count DESC
""").fetchdf()

,design_architecture,count
0,Parallel RCT,243105
1,Single-Arm,113575
2,Cohort,78704
3,Crossover RCT,33211
4,Non-Randomized Parallel,24018
5,Case-Control,18344
6,Case-Only,17263
7,Other Observational,13423
8,Randomized Single Group,8152
9,Non-Randomized Sequential,5758


### L4: Blinding Level

In [5]:
conn.execute("""
    SELECT blinding_level, COUNT(*) AS count
    FROM class.study_design
    WHERE blinding_level IS NOT NULL
    GROUP BY blinding_level
    ORDER BY count DESC
""").fetchdf()

,blinding_level,count
0,Open Label,247282
1,Single Blind,67662
2,Double Blind,60534
3,Quadruple Blind,40061
4,Triple Blind,29551


### L5: Purpose

In [6]:
conn.execute("""
    SELECT purpose, COUNT(*) AS count
    FROM class.study_design
    WHERE purpose IS NOT NULL
    GROUP BY purpose
    ORDER BY count DESC
""").fetchdf()

,purpose,count
0,TREATMENT,286471
1,PREVENTION,47697
2,OTHER,25861
3,SUPPORTIVE_CARE,25363
4,BASIC_SCIENCE,21396
5,DIAGNOSTIC,19173
6,HEALTH_SERVICES_RESEARCH,11862
7,SCREENING,4098
8,DEVICE_FEASIBILITY,1560
9,ECT,173


## 2. Innovative Features (L3)

In [7]:
total_features = conn.execute("SELECT COUNT(*) FROM class.innovative_features").fetchone()[0]
total_studies = conn.execute("SELECT COUNT(DISTINCT nct_id) FROM class.innovative_features").fetchone()[0]
all_studies = conn.execute("SELECT COUNT(*) FROM class.study_design").fetchone()[0]
pct = round(100 * total_studies / all_studies, 1)
print(f"Total feature rows: {total_features:,}")
print(f"Studies with at least one innovative feature: {total_studies:,} ({pct}%)")

Total feature rows: 20,346
Studies with at least one innovative feature: 14,377 (2.4%)


In [8]:
conn.execute("""
    SELECT feature_type, COUNT(DISTINCT nct_id) AS study_count
    FROM class.innovative_features
    GROUP BY feature_type
    ORDER BY study_count DESC
""").fetchdf()

,feature_type,study_count
0,adaptive,8443
1,pragmatic,3440
2,SMART,699
3,bayesian,619
4,platform,452
5,umbrella,388
6,basket,383
7,seamless,318
8,master protocol,292
9,N-of-1,158


### Source field breakdown

In [9]:
conn.execute("""
    SELECT feature_type, source_field, COUNT(*) AS count
    FROM class.innovative_features
    GROUP BY feature_type, source_field
    ORDER BY feature_type, count DESC
""").fetchdf()

,feature_type,source_field,count
0,AI-augmented design,description,6
1,AI-augmented design,official_title,3
2,AI-augmented design,brief_title,2
3,N-of-1,description,110
4,N-of-1,official_title,105
5,N-of-1,brief_title,48
6,N-of-1,keyword,43
7,SMART,description,520
8,SMART,official_title,327
9,SMART,brief_title,239


### Co-occurrence: studies with multiple features

In [10]:
conn.execute("""
    SELECT feature_count, COUNT(*) AS studies
    FROM (
        SELECT nct_id, COUNT(DISTINCT feature_type) AS feature_count
        FROM class.innovative_features
        GROUP BY nct_id
    )
    GROUP BY feature_count
    ORDER BY feature_count
""").fetchdf()

,feature_count,studies
0,1,13581
1,2,693
2,3,84
3,4,17
4,5,2


## 3. AI/ML Mentions (Phase 2B.1)

`class.ai_mentions` tracks studies that mention AI, machine learning, deep learning, NLP, etc. in their free-text fields. Independent from `innovative_features` — a study can reference AI/ML without having an AI-augmented *design* (e.g. using ML only for analysis).

In [11]:
ai_total = conn.execute("SELECT COUNT(*) FROM class.ai_mentions").fetchone()[0]
ai_studies = conn.execute("SELECT COUNT(DISTINCT nct_id) FROM class.ai_mentions").fetchone()[0]
all_studies = conn.execute("SELECT COUNT(*) FROM class.study_design").fetchone()[0]
ai_pct = round(100 * ai_studies / all_studies, 1)
print(f"AI/ML mention rows:        {ai_total:,}")
print(f"Studies with an AI mention: {ai_studies:,} ({ai_pct}%)")

AI/ML mention rows:        12,148
Studies with an AI mention: 5,827 (1.0%)


In [12]:
conn.execute("""
    SELECT ai_term, COUNT(DISTINCT nct_id) AS studies
    FROM class.ai_mentions
    GROUP BY ai_term
    ORDER BY studies DESC
""").fetchdf()

,ai_term,studies
0,artificial intelligence,3425
1,machine learning,2089
2,deep learning,859
3,neural network,407
4,large language model,219
5,natural language processing,151
6,computer vision,148
7,ChatGPT/GPT,124
8,reinforcement learning,100


## 4. Spot-Checks

Random sample of 10 studies per feature type for manual precision verification.

In [13]:
feature_types = conn.execute("""
    SELECT DISTINCT feature_type FROM class.innovative_features ORDER BY feature_type
""").fetchall()

for (ft,) in feature_types:
    print(f"\n{'='*80}")
    print(f"Feature: {ft}")
    print(f"{'='*80}")
    samples = conn.execute(f"""
        SELECT f.nct_id, f.source_field, f.matched_text, s.brief_title
        FROM class.innovative_features f
        JOIN raw.studies s ON f.nct_id = s.nct_id
        WHERE f.feature_type = '{ft}'
        ORDER BY random()
        LIMIT 10
    """).fetchdf()
    display(samples)


Feature: AI-augmented design


,nct_id,source_field,matched_text,brief_title
0,NCT01942356,description,reinforcement learning to titrate dos,"Evaluation of Closed-loop TIVA Propofol, Sufen..."
1,NCT07144618,description,AI-guided dosing,Investigating the Correlation Between Pre-Trea...
2,NCT03962400,official_title,Reinforcement Learning for the Dos,Reinforcement Learning for Warfarin Dosing
3,NCT06857344,official_title,Reinforcement Learning Based Control for Anest...,Reinforcement Learned Automatic Anesthesia Sys...
4,NCT05751993,official_title,Reinforcement Learning Tool for Individually T...,Piloting a Reinforcement Learning Tool for Ind...
5,NCT04440553,description,reinforcement learning algorithm in the adaptive,A Mobile App to Increase Physical Activity in ...
6,NCT03962400,brief_title,Reinforcement Learning for Warfarin Dos,Reinforcement Learning for Warfarin Dosing
7,NCT07228130,description,Reinforcement learning will be applied to the ...,Microrandomized Trial to Optimize Use of Burde...
8,NCT07099261,description,AI-driven design,A Comparative Clinical Study to Evaluate Crown...
9,NCT05751993,brief_title,Reinforcement Learning Tool for Individually T...,Piloting a Reinforcement Learning Tool for Ind...



Feature: N-of-1


,nct_id,source_field,matched_text,brief_title
0,NCT01258764,description,N-of-1,Hypertensive Ambulatory Trial to Compare the E...
1,NCT02142036,official_title,N-of-1,N-of-1 Trial: Actionable Target Identification...
2,NCT07150013,keyword,n of 1,Rett REVOLUTION Trial: An Exploratory Evaluati...
3,NCT07549685,official_title,N-of-1,Music Therapy in Patients With Breast Cancer (...
4,NCT03695263,official_title,N-of-1,Massive Individualized N-of-1 Experiments (MINEs)
5,NCT00131248,description,N of 1,Medical Treatment for Gastroesophageal Reflux ...
6,NCT01983592,brief_title,N-of-1,An N-of-1 Study of Homeopathic Treatment of Fa...
7,NCT06827041,official_title,N-of-1,Use of Phage Therapy for Treatment of a Peripr...
8,NCT05161182,official_title,N-of-1,Westlake N-of-1 Trials for Macronutrient Intak...
9,NCT04132011,official_title,N-of-1,Dosing Intervals of Opioid Medication for Chro...



Feature: SMART


...

,nct_id,source_field,matched_text,brief_title
0,NCT04787289,official_title,Pragmatic,A Comparison of 2 Standard Doses of Bevacizuma...
1,NCT03821844,brief_title,Pragmatic,Music & MEmory: A Pragmatic TRial for Nursing ...
2,NCT05900375,keyword,pragmatic,Decision Aid for Parents of Infants With UPJO
3,NCT04054648,keyword,Pragmatic,BedMed-Frail: Does the Potential Benefit of Be...
4,NCT01271985,description,pragmatic,Prevention of Cardiovascular Disease in Middle...
5,NCT03124316,description,pragmatic,Testing a Behavioural Approach to Improving Ca...
6,NCT04023643,official_title,Pragmatic,Pediatric Ventilation Weaning
7,NCT05306743,description,pragmatic,PRagmatic EVAluation of a Quality Improvement ...
8,NCT03896776,official_title,Pragmatic,A Pragmatic Trial of Two Strategies for Implem...
9,NCT03725384,description,pragmatic,Daily vs. Every Other Day Oral Iron Supplement...



Feature: seamless


,nct_id,source_field,matched_text,brief_title
0,NCT06169696,description,seamless,EMPOWER Early Feasibility Study: Non-invasive ...
1,NCT06702449,official_title,Seamless,A Study on the Safety and Immune Response of a...
2,NCT04232696,description,seamless,Neuspera's Implantable Sacral Nerve Stimulatio...
3,NCT07511920,description,seamless,A Multicenter Cohort Study of Duchenne and Bec...
4,NCT05457946,official_title,Seamless,Study to Evaluate the Immunogenicity and Safet...
5,NCT04222556,description,seamless,Evaluation of Thiwáhe Gluwáš'Akapi Substance U...
6,NCT05089656,description,seamless,Efficacy and Safety of Intrathecal OAV101 (AVX...
7,NCT06661291,description,seamless,Pip Care to Improve Surgical Patient Outcomes
8,NCT03044028,description,seamless,Neuromuscular Electrical Stimulation (NMES) fo...
9,NCT06947499,official_title,Seamless,A Phase II/III Study to Evaluate the Immunogen...



Feature: umbrella


,nct_id,source_field,matched_text,brief_title
0,NCT01998763,description,umbrella,Vitamin D Intervention Trial in Healthy Chines...
1,NCT02826408,description,umbrella,Effect of Proton Pump Inhibitors on Palpitations
2,NCT03010449,description,umbrella,LVR in Severe Emphysema Using Bronchoscopic Au...
3,NCT04180319,description,umbrella,EARCO REGISTRY. History Of Patients With Alpha...
4,NCT07050082,description,umbrella,Cystoinflation to Prevent Bladder Injury in Ca...
5,NCT06125522,keyword,Umbrella,TACTIVE-U: A Study to Learn About the Study Me...
6,NCT05474742,description,umbrella,The Nori Health App
7,NCT03555149,official_title,Umbrella,A Study Evaluating the Efficacy and Safety of ...
8,NCT06267430,description,umbrella,Learning by Heart: The Effectiveness of an EF ...
9,NCT07440290,official_title,Umbrella,DETERMINE Trial Treatment Arm 07: Dabrafenib i...


In [15]:
conn.close()